<center><a target="_blank" href="https://academy.constructor.org/"><img src=https://lh3.googleusercontent.com/d/1fypIr9T-7ntcsVQFmC2_iMPcsm7h8jXg width="500" style="background:none; border:none; box-shadow:none;" /></a> </center>
<hr />

# <h1 align="center"> Day-1-Descriptive-Statistics. Exercise session 4: Probability Computation in a Business Context </h1> </center>

<p style="margin-bottom:1cm;"></p>

_____

<center>Constructor Nexademy, 2026</center>


In [6]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd

Exercise Goal: Experiment with Bayes Law, compute frequencies and link them to probabilities.

Bayes’ Law states that the probability of an event A happening given that B happened is equal to the ratio of the probability of B happening given A happened times the probability of A happening over the probability of B happening…. or in short:

$$P(A|B) = \frac{P(B|A)P(A)}{P(B)}$$

Now. How can this help us infer anything in a business context? That’s what we’re gonna see in this exercise.

Let’s look at the Marketing data again (we already worked on this problem when looking at train/test splits). Here is the description of the problem:

You have been contacted by company B, that would like to increase their sales. They have recently done a marketing campaign where they sent a promotional email to their customer with either:

- a buy one get one offer
- a discount offer
- no offer.

They sent you the Marketing.csv file, with the following columns:

- recency: months since the last purchase made by customer
- history: total amount ever spent by the customer in the shop
- used_discount: whether the customer has already used a discount before
- used_bogo: whether the customer has already used a “buy one, get one” offer before
- zip_code: class of the zip code (the customer can be either Suburban, Urban or Rural)
- is_referral: whether the customer was acquired through a referal channel
- channel: channels that the customer is using (Phone, Web, Multichannel)
- offer: the offer sent ()
- conversion: 1=customer bought something, 0= customer did not buy anything.

In [11]:
# reading data from csv
df = pd.read_csv('/home/soraan/Desktop/exer 5/Marketing3.csv', index_col=[0])
print(df[['offer', 'conversion']].head())

             offer  conversion
0  Buy One Get One           0
1  Buy One Get One           0
2  Buy One Get One           1
3  Buy One Get One           1
4  Buy One Get One           0


In [12]:
p_conversion = df['conversion'].mean()
print(f"Overall Conversion Rate: {p_conversion:.2f}")

p_offer = df['offer'].value_counts(normalize=True)[1]
print("Percentage of each offer sent:")
print(p_offer)

Overall Conversion Rate: 0.15
Percentage of each offer sent:
0.332921875


/tmp/ipykernel_31891/4059244116.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  p_offer = df['offer'].value_counts(normalize=True)[1]


In [13]:
comparation = df.groupby('offer')['conversion'].mean()
print("\nConversion Rate by Offer:")
print(comparation)


Conversion Rate by Offer:
offer
Buy One Get One    0.151400
Discount           0.182757
No Offer           0.106167
Name: conversion, dtype: float64


## 1. Compute probabilities

**What is the overall probability of conversion for a random customer?**

$$P(conv) = \frac{N(conv)}{N(total)}$$

In [16]:
p_conv = df['conversion'].mean()
print(f"Overall Conversion Rate: {p_conv:.4f}")

print(f"\n the overall probability is : {p_conv * 100:.2f}%")


Overall Conversion Rate: 0.1468

 the overall probability is : 14.68%


**What is the probability that a randomly selected customer within the database has both been active whithin the last 6 months and has been receptive to the marketing campaign**

In [18]:
active_and_converted = df[(df['recency'] <= 6) & (df['conversion'] == 1)]
p_active_and_converted = len(active_and_converted) / len(df)
print(f"Probability of being active and converting: {p_active_and_converted:.4f}")
print(f"Probability of being active and converting: {p_active_and_converted * 100:.2f}%")

Probability of being active and converting: 0.0953
Probability of being active and converting: 9.53%


This is the probability of the customer **being active and subject to conversion**.

**Now, what is the probability of conversion of a customer that was active within the last 6 months?**

In [24]:
active_costumers = df[df['recency'] <= 6]

p_converted_given_active = active_costumers['conversion'].mean()
print(f"Probability of converting given active: {p_converted_given_active:.4f}")
print(f"Probability of converting given active: {p_converted_given_active * 100:.2f}%")

Probability of converting given active: 0.1666
Probability of converting given active: 16.66%


**The Marketing team is coming back with new numbers: the demographics has**
**changed and now they have:**

- 30% rural
- 50% urban
- 20% Surburban

How would you correct the global conversion rate?

In [27]:
print(df['zip_code'].unique())


['Surburban' 'Rural' 'Urban']


In [28]:
conv_rates = df.groupby('zip_code')['conversion'].mean()

net_weights = {
    'Rural': 0.30,
    'Surburban': 0.50,
    'Urban': 0.20
}
corrected_gloval_conv = (
    conv_rates['Rural'] * net_weights['Rural'] +
    conv_rates['Surburban'] * net_weights['Surburban'] +
    conv_rates['Urban'] * net_weights['Urban']

)
print(f"Corrected Global Conversion Rate: {corrected_gloval_conv:.4f}")
print(f"Corrected Global Conversion Rate: {corrected_gloval_conv * 100:.2f}%")

Corrected Global Conversion Rate: 0.1542
Corrected Global Conversion Rate: 15.42%


## 2. Bayes law: Integrate prior knowledge

Let's say you built a predictive model out of your data: you made a train/test
split as recommended in exercise 1, then you applied a classification algorithm,
and now you have a model that can predict, from history, recency, zip_code, ...
whether a marketing sollicitation is going to result in a conversion.

During the test phase you obtain the following counts:

|                          | Customers than converted | Customers that did not convert |
|--------------------------|-----------------|---------------------|
| Predicted: conv = 1  | 100             | 10                  |
| Predicted: conv = 0  | 4               | 1000                |

now the marketing team is giving you new data, and for a given customer, your
model has predicted a conversion.

**Apply the Bayes theorem to compute the probability that they'll convert.**

In [29]:

tp = 100  
fp = 10 
fn = 4  
tn = 1000 


total = tp + fp + fn + tn
p_actual_conv = (tp + fn) / total  # P(A): Real conversion rate in test data
p_predicted_conv = (tp + fp) / total  # P(B): How often the model says "1"
p_pred_given_actual = tp / (tp + fn)  # P(B|A): Model accuracy for real buyers

p_conv_given_pred = (p_pred_given_actual * p_actual_conv) / p_predicted_conv

print(f"The probability that the customer actually converts is: {p_conv_given_pred:.4f}")
print(f"In percentage: {p_conv_given_pred * 100:.2f}%")

The probability that the customer actually converts is: 0.9091
In percentage: 90.91%


**Now the Marketing team is contacting you again and says that historically,**
**their conversion rate has actually been of 5%. How would you update your probability?**

In [30]:
tp = 100
fp = 10
fn = 4
tn = 1000

p_pred_given_actual = tp / (tp + fn) 
p_pred_given_not_actual = fp / (fp + tn)

p_actual_new = 0.05  # The 5% rate
p_not_actual_new = 1 - p_actual_new

numerator = p_pred_given_actual * p_actual_new
denominator = (p_pred_given_actual * p_actual_new) + (p_pred_given_not_actual * p_not_actual_new)

updated_prob = numerator / denominator

print(f"Updated Probability with 5% market rate: {updated_prob:.4f}")
print(f"In percentage: {updated_prob * 100:.2f}%")

Updated Probability with 5% market rate: 0.8364
In percentage: 83.64%
